In [ ]:
import json
import time
import datetime
import requests
import pandas as pd
import os
import re
import matplotlib.pyplot as plt
from datetime import datetime


## Load country regrex

In [ ]:
# Confirm current working directory
cwd = os.getcwd()
print(f"Current working directory: {cwd}")

In [ ]:
regrex_file = f"{cwd}/query_regrex_all.csv"

In [ ]:
# Assumes 'query_regrex.csv' has columns: ISO3, regex
df_regex = pd.read_csv(regrex_file, dtype=str, encoding="latin1")

# Build a dict:  country_info["USA"]  ->  {"regex":   r"(\b)(united...)", 
#                                          "region": "usa"}
country_info = (
    df_regex
    .set_index('iso3')[['regex', 'region_code']]
    .to_dict('index')
)

## Query templates

In [ ]:
def inject_country(template: str, regex: str, region_code: str) -> str:
    return (template
            .replace('__COUNTRY_REGEX__', regex)
            .replace('__REGION_CODE__',  region_code.lower()))  # region-codes are stored lowercase

In [ ]:
api_key = ''
key_headers = {
    'user-key': 'CyYzoIStUsa5wgdSTeSOglpPU4gaFxJV', # Enter your API key here
    'content-type': "application/json",
    'cache-control': "no-cache",
    'X-API-VERSION' : "2.0"
}

In [ ]:
# ============================================================
# FPU SUB-CATEGORY KEYWORDS
# Extraction rule: Tax OR Expenditure OR Debt
# Article labels are non-mutually-exclusive (multi-label).
# ============================================================

TAX_PATTERN = (
    r"tax"
    r"|taxed"
    r"|taxation"
    r"|taxes"
)

EXPENDITURE_PATTERN = (
    r"government\W+spending"
    r"|federal\W+spending"
    r"|public\W+spending"
    r"|government\W+expenditure\w*"
    r"|federal\W+expenditure\w*"
    r"|public\W+expenditure\w*"
    r"|defense\W+spending"
    r"|defence\W+spending"
    r"|military\W+spending"
    r"|pension\W+reform\w*"
    r"|pension\W+expenditure\w*"
    r"|healthcare\W+expenditure\w*"
    r"|medical\W+care\W+expenditure\w*"
    r"|social\W+expenditure\w*"
    r"|social\W+safety\W+net\w*"
    r"|public\W+investment\w*"
    r"|subsidy"
    r"|subsidies"
    r"|subsidised"
    r"|subsidized"
    r"|entitlement\W+spending"
    r"|social\W+security"
    r"|fiscal\W+stimulus"
    r"|social\W+protection"
    r"|social\W+security\W+expenditure\w*"
    r"|defense\W+expenditure\w*"
    r"|defence\W+expenditure\w*"
    r"|military\W+expenditure\w*"
    r"|pension\W+spending"
    r"|health\W+care\W+expenditure\w*"
    r"|health\W+spending"
    r"|healthcare\W+spending"
    r"|medical\W+care\W+spending"
    r"|social\W+spending"
    r"|fiscal\W+cliff"
)

DEBT_PATTERN = (
    r"federal\W+debt"
    r"|government\W+debt"
    r"|national\W+debt"
    r"|public\W+debt"
    r"|sovereign\W+debt"
    r"|debt\W+burden\w*"
    r"|debt\W+repay\w*"
    r"|debt\W+sustainability"
    r"|government\W+borrowing"
    r"|sovereign\W+borrowing"
    r"|public\W+borrowing"
    r"|national\W+borrowing"
    r"|debt\W+consolidation"
    r"|sovereign\W+default\w*"
    r"|debt\W+default\w*"
    r"|debt\W+restructur\w*"
    r"|foreign\W+debt"
    r"|external\W+debt"
    r"|government\W+bond\w*"
    r"|sovereign\W+bond\w*"
    r"|sovereign\W+yield\w*"
    r"|balanced\W+budget"
    r"|balance\W+the\W+budget"
    r"|budget\W+deficit\w*"
    r"|government\W+deficit\w*"
    r"|national\W+deficit\w*"
    r"|federal\W+deficit\w*"
    r"|budget\W+gap\w*"
    r"|debt\W+ceiling\w*"
    r"|sovereign\W+risk"
    r"|fiscal\W+deficit\w*"
)

# IMPORTANT: OR across the three fiscal categories.
FISCAL_PATTERN = rf"(?:{TAX_PATTERN}|{EXPENDITURE_PATTERN}|{DEBT_PATTERN})"

POLICY_PATTERN = (
    r"legislat\w*|minister|policies|policy|regulat\w*|government|"
    r"congress\w*|parliament|national\W+assembly|ministry"
)

UNCERTAINTY_PATTERN = (
    r"doubt|doubtful|risk\w*|unresolved|unsettled|dubious|undetermined|"
    r"undecided|unpredictable|precarious|ambiguous|uncertain\w*|volatilit\w*"
)

ECONOMIC_PATTERN = r"economic|economy"

RECESSION_PATTERN = (
    r"recession\w*|"
    r"econom\w*\W+(?:\w+\W+){0,5}?(?:downturn\w*|slowdown\w*|slow\w*\W+down|contract\w*|slump\w*|collapse\w*|stagnat\w*|depress\w*|meltdown)|"
    r"(?:downturn\w*|slowdown\w*|slow\w*\W+down|contract\w*|slump\w*|stagnat\w*|depress\w*|meltdown|collapse\w*)\W+(?:\w+\W+){0,5}?econom\w*|"
    r"GDP\W+(?:\w+\W+){0,5}?(?:contract\w*|declin\w*|shrink\w*)|"
    r"(?:contract\w*|declin\w*|shrink\w*)\W+(?:\w+\W+){0,5}?GDP|"
    r"negative\W+GDP\W+growth|"
    r"(?:output|manufacturing)\W+(?:\w+\W+){0,5}?(?:contract\w*|fall\w*|slump\w*)|"
    r"(?:contract\w*|fall\w*|slump\w*)\W+(?:\w+\W+){0,5}?(?:output|manufacturing)|"
    r"financial\W+crisis|economic\W+crisis|banking\W+crisis|credit\W+crisis|"
    r"currency\W+crisis|credit\W+crunch|liquidity\W+crisis|output\W+collapse|market\W+collapse"
)

SOURCE_PATTERN = (
    r"(^|,)(ctbonl|sfc|j|dal|usatonl|wp|gma|absr|wnsa|wnt|nlne|thwk|wnsu|"
    r"abcaze|bbcmna|bbcmre|bbcrnd|bbcpsm|sxmn|cbst|cbsa|fcnt|ntln|toda|t|"
    r"grdn|telem|lba|glob|ec|stel|bbcap|bbcmap|bbcapp|bbccau|bbcca|bbceup|"
    r"bbcsup|bbcmm|bbcmep|bbcmnf|bbcsap|bbcukb|aprs)($|,)"
)

TEXT_EXPR = "CONCAT(title, ' ', IFNULL(snippet, ''), ' ', IFNULL(body, ''), ' ', IFNULL(section, ''))"


In [ ]:
# ============================================================
# FPU QUERY BUILDERS
# Aggregate extraction rule: Tax OR Expenditure OR Debt
# Subcategory queries use the same common FPU conditions and differ
# only in the fiscal-category pattern.
# ============================================================

def build_fpu_where(fiscal_pattern: str, exclude_recession: bool = False) -> str:
    where = (
        f"REGEXP_CONTAINS({TEXT_EXPR}, r'(?i)(\b)(?:{POLICY_PATTERN})(\b)')"
        f" AND REGEXP_CONTAINS({TEXT_EXPR}, r'(?i)(\b)(?:{UNCERTAINTY_PATTERN})(\b)')"
        f" AND REGEXP_CONTAINS({TEXT_EXPR}, r'(?i)(\b)(?:{fiscal_pattern})(\b)')"
        f" AND REGEXP_CONTAINS({TEXT_EXPR}, r'(?i)(\b)(?:{ECONOMIC_PATTERN})(\b)')"
        " AND REGEXP_CONTAINS(CONCAT(title, ' ', IFNULL(snippet, '')), r'(?i)__COUNTRY_REGEX__')"
        f" AND REGEXP_CONTAINS(restrictor_codes, r'(?i){SOURCE_PATTERN}')"
        " AND LOWER(region_codes) LIKE '%,__REGION_CODE__,%'"
    )
    if exclude_recession:
        where += f" AND NOT REGEXP_CONTAINS({TEXT_EXPR}, r'(?i)(\b)(?:{RECESSION_PATTERN})(\b)')"
    return where

# Aggregate numerator: ANY of the three categories (OR)
numerator_base_where = build_fpu_where(FISCAL_PATTERN, exclude_recession=False)

# Recession-excluded aggregate numerator: same OR rule
numerator_rece_where = build_fpu_where(FISCAL_PATTERN, exclude_recession=True)

# Category-specific numerators. These are separate count queries used to
# construct Tax / Expenditure / Debt FPU while allowing overlap across categories.
tax_where = build_fpu_where(TAX_PATTERN, exclude_recession=False)
expenditure_where = build_fpu_where(EXPENDITURE_PATTERN, exclude_recession=False)
debt_where = build_fpu_where(DEBT_PATTERN, exclude_recession=False)


In [ ]:
# Query logic check
#
# Aggregate FPU article count:
#   Policy AND Uncertainty AND (Tax OR Expenditure OR Debt) AND Economy ...
#
# Subcategory counts:
#   Policy AND Uncertainty AND Tax AND Economy ...
#   Policy AND Uncertainty AND Expenditure AND Economy ...
#   Policy AND Uncertainty AND Debt AND Economy ...
#
# A single article may therefore be counted in more than one subcategory.
# Do NOT calculate aggregate numerator as tax_count + expenditure_count + debt_count.
print("Aggregate fiscal rule: Tax OR Expenditure OR Debt")
print("Subcategory counts preserve overlap across Tax / Expenditure / Debt.")


In [ ]:
# Denominator is unchanged: all qualifying news for the country/source/region.
denominator_where = (
    "REGEXP_CONTAINS(CONCAT(title, ' ', IFNULL(snippet, '')), r'(?i)__COUNTRY_REGEX__')"
    f" AND REGEXP_CONTAINS(restrictor_codes, r'(?i){SOURCE_PATTERN}')"
    " AND LOWER(region_codes) LIKE '%,__REGION_CODE__,%'"
)


### Article-level Tax / Expenditure / Debt labels

The current Dow Jones Analytics query used below returns aggregated `publication_datetime` + `count` results. Therefore the existing country loop continues to produce the aggregate FPU series exactly as before, but with the revised `(Tax OR Expenditure OR Debt)` extraction rule.

The helper below is for article-level records when `title`, `snippet`, `body`, and/or `section` are available. Labels are **multi-label**, so the same article may have `tax=1` and `debt=1` at the same time. No `not_assigned` category is created.


In [ ]:
# ============================================================
# ARTICLE-LEVEL MULTI-LABELING HELPERS
# No not_assigned category.
# ============================================================

TAX_RE = re.compile(rf"(?i)(\b)(?:{TAX_PATTERN})(\b)")
EXPENDITURE_RE = re.compile(rf"(?i)(\b)(?:{EXPENDITURE_PATTERN})(\b)")
DEBT_RE = re.compile(rf"(?i)(\b)(?:{DEBT_PATTERN})(\b)")


def _matched_terms(text, regex):
    if pd.isna(text):
        return []
    return sorted({m.group(0).strip() for m in regex.finditer(str(text))})


def label_fpu_articles(article_df: pd.DataFrame) -> pd.DataFrame:
    """
    Add non-mutually-exclusive Tax / Expenditure / Debt labels.

    Expected article text columns (missing columns are treated as blank):
        title, snippet, body, section

    Output labels:
        tax, expenditure, debt   (0/1)
        tax_keywords, expenditure_keywords, debt_keywords
        n_categories, category_combination
        tax_expenditure, tax_debt, expenditure_debt, all_three
    """
    df = article_df.copy()

    for col in ["title", "snippet", "body", "section"]:
        if col not in df.columns:
            df[col] = ""

    df["article_text"] = (
        df["title"].fillna("").astype(str) + " "
        + df["snippet"].fillna("").astype(str) + " "
        + df["body"].fillna("").astype(str) + " "
        + df["section"].fillna("").astype(str)
    )

    df["tax"] = df["article_text"].str.contains(TAX_RE, na=False).astype("int8")
    df["expenditure"] = df["article_text"].str.contains(EXPENDITURE_RE, na=False).astype("int8")
    df["debt"] = df["article_text"].str.contains(DEBT_RE, na=False).astype("int8")

    df["tax_keywords"] = df["article_text"].apply(lambda x: "; ".join(_matched_terms(x, TAX_RE)))
    df["expenditure_keywords"] = df["article_text"].apply(lambda x: "; ".join(_matched_terms(x, EXPENDITURE_RE)))
    df["debt_keywords"] = df["article_text"].apply(lambda x: "; ".join(_matched_terms(x, DEBT_RE)))

    df["n_categories"] = df[["tax", "expenditure", "debt"]].sum(axis=1).astype("int8")

    def _combination(row):
        labels = []
        if row["tax"] == 1:
            labels.append("Tax")
        if row["expenditure"] == 1:
            labels.append("Expenditure")
        if row["debt"] == 1:
            labels.append("Debt")
        return " + ".join(labels) if labels else "Check"

    df["category_combination"] = df.apply(_combination, axis=1)

    df["tax_expenditure"] = ((df["tax"] == 1) & (df["expenditure"] == 1)).astype("int8")
    df["tax_debt"] = ((df["tax"] == 1) & (df["debt"] == 1)).astype("int8")
    df["expenditure_debt"] = ((df["expenditure"] == 1) & (df["debt"] == 1)).astype("int8")
    df["all_three"] = ((df["tax"] == 1) & (df["expenditure"] == 1) & (df["debt"] == 1)).astype("int8")

    return df


In [ ]:
def build_payload(where_clause: str) -> dict:
    """Return the dict to pass to requests.post."""
    return {
        "query": {
            "where": where_clause,
            "top": -1,
            "format": "json"      # JSON easier to parse than CSV; change if you prefer
        }
    }

## Functions to one query

In [ ]:
analytics_url = "https://api.dowjones.com/analytics"

In [ ]:
def run_factiva_query(payload: dict) -> dict:
    """
    Submit one Factiva Analytics query, poll the SAME job until finished,
    and return the final JSON.

    Important handling:
    - 300-second request timeout
    - retries temporary polling 404/429/5xx responses
    - Dow Jones code 5513 ("Empty result set") is NOT treated as a failure
      -> returns an empty results list so the corresponding count becomes 0
    - never resubmits a second analytics job while the current one is active
    - preserves the original 6-minute courtesy/rate-limit spacing
    """

    start = time.time()

    def _wait_for_rate_limit():
        elapsed = time.time() - start
        wait = max(0, 360 - elapsed)
        if wait:
            print(f"    Waiting {int(wait)} s for rate limit …")
            time.sleep(wait)

    def _empty_result_json():
        return {
            "data": {
                "attributes": {
                    "current_state": "JOB_STATE_DONE",
                    "results": []
                }
            }
        }

    def _is_empty_result_response(response):
        if response.status_code != 404:
            return False

        text_lower = response.text.lower()

        if "empty result set" in text_lower:
            return True

        try:
            body = response.json()
            for err in body.get("errors", []):
                if str(err.get("code", "")) == "5513":
                    return True
        except Exception:
            pass

        return False

    # --------------------------------------------------------
    # 1) SUBMIT ONE ANALYTICS JOB
    # --------------------------------------------------------
    resp = requests.post(
        analytics_url,
        data=json.dumps(payload),
        headers=key_headers,
        timeout=300
    )

    print(f"    POST status: {resp.status_code}")

    if resp.status_code >= 400:
        print("    API submit error:")
        print(resp.text[:2000])

    resp.raise_for_status()

    job = resp.json()

    job_url = job["links"]["self"]
    job_state = job["data"]["attributes"]["current_state"]

    print(f"    Job URL: {job_url}")
    print(f"    Initial state: {job_state}")

    latest_json = job

    # --------------------------------------------------------
    # Helper: GET the SAME job URL safely
    # --------------------------------------------------------
    def get_same_job(max_attempts=12):
        for attempt in range(1, max_attempts + 1):
            try:
                r = requests.get(
                    job_url,
                    headers=key_headers,
                    timeout=300
                )
            except requests.exceptions.RequestException as exc:
                if attempt == max_attempts:
                    raise

                wait_seconds = min(10 * attempt, 60)
                print(
                    f"    Poll request error ({type(exc).__name__}); "
                    f"retrying same job in {wait_seconds}s "
                    f"[{attempt}/{max_attempts}]"
                )
                time.sleep(wait_seconds)
                continue

            # Dow Jones uses 404 / code 5513 for a valid zero-match query.
            if _is_empty_result_response(r):
                print(
                    "    Empty result set (Dow Jones code 5513) "
                    "→ treating this count as 0."
                )
                return "empty", None

            # Normal success
            if 200 <= r.status_code < 300:
                return "ok", r

            # Temporary/not-yet-visible or transient server/rate-limit errors
            if r.status_code in {404, 429, 500, 502, 503, 504}:
                if attempt == max_attempts:
                    print("    Final polling error response:")
                    print(r.text[:2000])
                    r.raise_for_status()

                wait_seconds = min(10 * attempt, 60)
                print(
                    f"    Poll status {r.status_code}; "
                    f"retrying SAME job in {wait_seconds}s "
                    f"[{attempt}/{max_attempts}]"
                )
                time.sleep(wait_seconds)
                continue

            # Other errors stop immediately
            print(f"    Poll status: {r.status_code}")
            print(r.text[:2000])
            r.raise_for_status()

        raise RuntimeError("Polling retry loop ended unexpectedly.")

    # --------------------------------------------------------
    # 2) POLL UNTIL DONE / FAILED / EMPTY
    # --------------------------------------------------------
    while job_state not in {"JOB_STATE_DONE", "JOB_STATE_FAILED"}:
        time.sleep(10)

        status, poll = get_same_job()

        if status == "empty":
            _wait_for_rate_limit()
            return _empty_result_json()

        latest_json = poll.json()
        job_state = latest_json["data"]["attributes"]["current_state"]
        print(f"    Job state: {job_state}")

    # --------------------------------------------------------
    # 3) FETCH FINAL RESULT
    # --------------------------------------------------------
    if job_state == "JOB_STATE_DONE":
        status, final = get_same_job()

        if status == "empty":
            final_json = _empty_result_json()
        else:
            final_json = final.json()

    else:
        final_json = latest_json
        print("    Factiva job failed:")
        print(json.dumps(final_json, indent=2)[:3000])
        raise RuntimeError("Factiva Analytics job returned JOB_STATE_FAILED.")

    # --------------------------------------------------------
    # 4) COURTESY / RATE-LIMIT WAIT
    # --------------------------------------------------------
    _wait_for_rate_limit()

    return final_json


## Loop over countries

In [ ]:
records = []


def _prepare_count_df(rows, output_count_name):
    """Convert one Factiva Analytics count response to date + named count."""
    df = pd.DataFrame(rows)

    if "publication_datetime" not in df.columns:
        return pd.DataFrame(columns=["publication_datetime", output_count_name])

    df["publication_datetime"] = pd.to_datetime(df["publication_datetime"])

    if "count" not in df.columns:
        df["count"] = 0

    # Guard against duplicate date rows in an API response.
    df = (
        df.groupby("publication_datetime", as_index=False)["count"]
          .sum()
          .rename(columns={"count": output_count_name})
    )
    return df


for iso, info in country_info.items():
    print(f"=== {iso} ===")

    regex = info['regex']
    region_code = info['region_code']

    # Inject country-specific terms into all query templates.
    where_num_base = inject_country(numerator_base_where, regex, region_code)
    where_num_rece = inject_country(numerator_rece_where, regex, region_code)
    where_tax = inject_country(tax_where, regex, region_code)
    where_expenditure = inject_country(expenditure_where, regex, region_code)
    where_debt = inject_country(debt_where, regex, region_code)
    where_den = inject_country(denominator_where, regex, region_code)

    # Build payloads.
    payloads = {
        "numerator_base_count": build_payload(where_num_base),
        "numerator_rece_count": build_payload(where_num_rece),
        "tax_count": build_payload(where_tax),
        "expenditure_count": build_payload(where_expenditure),
        "debt_count": build_payload(where_debt),
        "denominator_count": build_payload(where_den),
    }

    # Run each count query separately. This preserves multi-category overlap:
    # one article may contribute to tax_count and debt_count simultaneously.
    dfs = []
    for count_name, payload in payloads.items():
        print(f"  -> {count_name}")
        result_json = run_factiva_query(payload)
        rows = result_json["data"]["attributes"].get("results", [])
        dfs.append(_prepare_count_df(rows, count_name))

    # Outer-merge all date-level counts.
    merged = None
    for df_piece in dfs:
        if merged is None:
            merged = df_piece.copy()
        else:
            merged = pd.merge(
                merged,
                df_piece,
                on="publication_datetime",
                how="outer"
            )

    if merged is None or merged.empty:
        merged = pd.DataFrame(columns=[
            "publication_datetime",
            "numerator_base_count",
            "numerator_rece_count",
            "tax_count",
            "expenditure_count",
            "debt_count",
            "denominator_count"
        ])

    count_cols = [
        "numerator_base_count",
        "numerator_rece_count",
        "tax_count",
        "expenditure_count",
        "debt_count",
        "denominator_count"
    ]

    for col in count_cols:
        if col not in merged.columns:
            merged[col] = 0

    merged[count_cols] = merged[count_cols].fillna(0)
    merged["iso3"] = iso
    records.append(merged)


## Final Dataframe

In [ ]:
final_df_raw = pd.concat(records, ignore_index=True)
# Optional: sort
final_df_raw = final_df_raw.sort_values(["iso3", "publication_datetime"])

print("\n=== SAMPLE OUTPUT ===")
print(final_df_raw.head())

In [ ]:
# 1. Ensure publication_datetime is datetime
final_df_raw['publication_datetime'] = pd.to_datetime(final_df_raw['publication_datetime'])

In [ ]:
# keep everything on or after Jan 1, 1995 to follow the IMF paper for period analysis since a large number of countries not having good coverage before that
final_df = final_df_raw[
    final_df_raw['publication_datetime'] >= pd.Timestamp('1995-01-01')
]

# Alternatively, you can comment out the above line and use the following line to keep everything

In [ ]:
# make it a real copy
final_df = final_df_raw.copy()

### Quarterly

In [ ]:
# 2) Group to quarters
quarterly_count_cols = [
    'numerator_base_count',
    'numerator_rece_count',
    'tax_count',
    'expenditure_count',
    'debt_count',
    'denominator_count'
]

quarterly = (
    final_df
    .groupby([
        'iso3',
        pd.Grouper(key='publication_datetime', freq='QE')
    ])[quarterly_count_cols]
    .sum()
    .reset_index()
)

# 3) Raw quarterly indices. The same denominator is used for all components.
quarterly['FPU_base_raw'] = quarterly['numerator_base_count'] / quarterly['denominator_count']
quarterly['FPU_rece_raw'] = quarterly['numerator_rece_count'] / quarterly['denominator_count']
quarterly['Tax_FPU_raw'] = quarterly['tax_count'] / quarterly['denominator_count']
quarterly['Expenditure_FPU_raw'] = quarterly['expenditure_count'] / quarterly['denominator_count']
quarterly['Debt_FPU_raw'] = quarterly['debt_count'] / quarterly['denominator_count']


def _z100(s):
    sd = s.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return pd.Series(pd.NA, index=s.index, dtype='Float64')
    return (s - s.mean()) / sd + 100

for raw, transformed in {
    'FPU_base_raw': 'FPU_base_transform',
    'FPU_rece_raw': 'FPU_rece_transform',
    'Tax_FPU_raw': 'Tax_FPU_transform',
    'Expenditure_FPU_raw': 'Expenditure_FPU_transform',
    'Debt_FPU_raw': 'Debt_FPU_transform'
}.items():
    quarterly[transformed] = quarterly.groupby('iso3')[raw].transform(_z100)

quarterly.rename(columns={'publication_datetime': 'quarter_end'}, inplace=True)
quarterly['quarter'] = quarterly['quarter_end'].dt.to_period('Q')


In [ ]:
# 5) Inspect
print(quarterly.head())

### Monthly

In [ ]:
# Monthly raw indices using a common denominator.
# IMPORTANT: aggregate numerator_base_count comes from the OR query and is NOT
# tax_count + expenditure_count + debt_count because subcategory overlap is allowed.
final_df['FPU_base_raw'] = final_df['numerator_base_count'] / final_df['denominator_count']
final_df['FPU_rece_raw'] = final_df['numerator_rece_count'] / final_df['denominator_count']
final_df['Tax_FPU_raw'] = final_df['tax_count'] / final_df['denominator_count']
final_df['Expenditure_FPU_raw'] = final_df['expenditure_count'] / final_df['denominator_count']
final_df['Debt_FPU_raw'] = final_df['debt_count'] / final_df['denominator_count']


def _normalize_monthly(s):
    sd = s.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return pd.Series(pd.NA, index=s.index, dtype='Float64')
    return (s - s.mean()) / sd + 100

for raw, transformed in {
    'FPU_base_raw': 'FPU_base_transform',
    'FPU_rece_raw': 'FPU_rece_transform',
    'Tax_FPU_raw': 'Tax_FPU_transform',
    'Expenditure_FPU_raw': 'Expenditure_FPU_transform',
    'Debt_FPU_raw': 'Debt_FPU_transform'
}.items():
    final_df[transformed] = final_df.groupby('iso3')[raw].transform(_normalize_monthly)

print("Monthly category count columns:", [
    c for c in ['tax_count', 'expenditure_count', 'debt_count']
    if c in final_df.columns
])


In [ ]:
# ============================================================
# MASTER-DATASET QC FIELDS
# ============================================================
# Category counts are intentionally non-mutually-exclusive.
# Therefore, their sum can exceed the aggregate OR numerator.

final_df['subcategory_sum'] = final_df[
    ['tax_count', 'expenditure_count', 'debt_count']
].sum(axis=1)

final_df['subcategory_sum_minus_aggregate'] = (
    final_df['subcategory_sum'] - final_df['numerator_base_count']
)

# Logical checks: each component is a subset of the aggregate OR query.
final_df['qc_tax_le_aggregate'] = (
    final_df['tax_count'] <= final_df['numerator_base_count']
)
final_df['qc_expenditure_le_aggregate'] = (
    final_df['expenditure_count'] <= final_df['numerator_base_count']
)
final_df['qc_debt_le_aggregate'] = (
    final_df['debt_count'] <= final_df['numerator_base_count']
)
final_df['qc_denominator_positive'] = final_df['denominator_count'] > 0

qc = final_df[[
    'iso3', 'publication_datetime',
    'numerator_base_count', 'tax_count', 'expenditure_count', 'debt_count',
    'subcategory_sum', 'subcategory_sum_minus_aggregate',
    'qc_tax_le_aggregate', 'qc_expenditure_le_aggregate',
    'qc_debt_le_aggregate', 'qc_denominator_positive'
]].copy()

print(qc['subcategory_sum_minus_aggregate'].describe())
print(
    'Rows with category overlap:',
    (qc['subcategory_sum_minus_aggregate'] > 0).sum()
)


In [ ]:
# ============================================================
# EXPORT RAW MASTER INPUT FOR CLEANING NOTEBOOK
# ============================================================
# The monthly and quarterly sheets retain counts, raw ratios, normalized
# diagnostic indices, and overlap fields. 02_Clean_GFPU_MasterDataset.ipynb
# will apply the coverage filter and create the publication-ready master file.

monthly_master_cols = [
    'iso3', 'publication_datetime',
    'denominator_count',
    'numerator_base_count', 'numerator_rece_count',
    'tax_count', 'expenditure_count', 'debt_count',
    'subcategory_sum', 'subcategory_sum_minus_aggregate',
    'FPU_base_raw', 'FPU_rece_raw',
    'Tax_FPU_raw', 'Expenditure_FPU_raw', 'Debt_FPU_raw',
    'FPU_base_transform', 'FPU_rece_transform',
    'Tax_FPU_transform', 'Expenditure_FPU_transform', 'Debt_FPU_transform'
]
monthly_master = final_df[[c for c in monthly_master_cols if c in final_df.columns]].copy()

# Add the same overlap diagnostics to quarterly output.
quarterly['subcategory_sum'] = quarterly[
    ['tax_count', 'expenditure_count', 'debt_count']
].sum(axis=1)
quarterly['subcategory_sum_minus_aggregate'] = (
    quarterly['subcategory_sum'] - quarterly['numerator_base_count']
)

quarterly_master_cols = [
    'iso3', 'quarter_end', 'quarter',
    'denominator_count',
    'numerator_base_count', 'numerator_rece_count',
    'tax_count', 'expenditure_count', 'debt_count',
    'subcategory_sum', 'subcategory_sum_minus_aggregate',
    'FPU_base_raw', 'FPU_rece_raw',
    'Tax_FPU_raw', 'Expenditure_FPU_raw', 'Debt_FPU_raw',
    'FPU_base_transform', 'FPU_rece_transform',
    'Tax_FPU_transform', 'Expenditure_FPU_transform', 'Debt_FPU_transform'
]
quarterly_master = quarterly[[c for c in quarterly_master_cols if c in quarterly.columns]].copy()

output_file = os.path.join(cwd, 'outputs', 'FPU_Factiva_raw.xlsx')
os.makedirs(os.path.dirname(output_file), exist_ok=True)

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    monthly_master.to_excel(writer, index=False, sheet_name='monthly')
    quarterly_master.to_excel(writer, index=False, sheet_name='quarterly')
    qc.to_excel(writer, index=False, sheet_name='qc')
    final_df_raw.to_excel(writer, index=False, sheet_name='all_periods')

print(f'Export complete: {output_file}')
print('Monthly master columns:')
print(monthly_master.columns.tolist())


In [ ]:
# Generate the filename with the current date for archive
current_datetime = datetime.now().strftime("%m%d%Y_%H%M")
archive_dir = f"{cwd}/Archives"

# Create Archives directory if it doesn't exist
os.makedirs(archive_dir, exist_ok=True)

archive_filename = f"{archive_dir}/FPU_Factiva_{current_datetime}.xlsx"

In [ ]:
# Save an archive copy using the same master structure
with pd.ExcelWriter(archive_filename, engine='openpyxl') as writer:
    monthly_master.to_excel(writer, index=False, sheet_name='monthly')
    quarterly_master.to_excel(writer, index=False, sheet_name='quarterly')
    qc.to_excel(writer, index=False, sheet_name='qc')

print(f'Export complete: {archive_filename}')


## Line charts

In [ ]:
country_iso3 = 'BEL'

In [ ]:
# 4. Plot for the United States
df_country = final_df[final_df['iso3'] == country_iso3].sort_values('publication_datetime')

plt.figure()
plt.plot(df_country['publication_datetime'], df_country['FPU_base_transform'])
plt.xlabel('Publication Date')
plt.ylabel('FPU Index')
plt.title(f'Fiscal Policy Uncertainty for {country_iso3}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Plot for the United States
df_country = final_df[final_df['iso3'] == country_iso3].sort_values('publication_datetime')

plt.figure()
plt.plot(df_country['publication_datetime'], df_country['FPU_rece_transform'])
plt.xlabel('Publication Date')
plt.ylabel('FPU Index')
plt.title(f'Fiscal Policy Uncertainty (Excluding Recessions Related Words) for {country_iso3}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()